In [ ]:
from datasets import load_dataset

klue_mrc_train = load_dataset('klue', 'mrc', split='train')
klue_mrc_train[0]

In [ ]:
from sentence_transformers import SentenceTransformer

# 임베딩 모델
sentence_model = SentenceTransformer('shangrilar/klue-roberta-base-klue-sts')

In [ ]:
from datasets import load_dataset

klue_mrc_train = load_dataset('klue', 'mrc', split='train')
klue_mrc_test = load_dataset('klue', 'mrc', split='validation')

df_train = klue_mrc_train.to_pandas()
df_test = klue_mrc_test.to_pandas()

df_train = df_train[['title', 'question', 'context']]
df_test = df_test[['title', 'question', 'context']]

In [ ]:
def add_ir_context(df):
    irrelevant_contexts = []
    for idx, row in df.iterrows():
        title = row['title']
        irrelevant_contexts.append(df.query(f"title != '{title}'").sample(n=1)['context'].values[0])
    df['irrelevant_context'] = irrelevant_contexts
    return df

df_train_ir = add_ir_context(df_train)
df_test_ir = add_ir_context(df_test)

In [ ]:
# 성능 평가 데이터 생성.
from sentence_transformers import InputExample

examples = []
for idx, row in df_test_ir.iterrows():
    examples.append(InputExample(texts=[row['question'], row['context']], label=1)) # question, context 컬럼은 서로 관련 있음.
    examples.append(InputExample(texts=[row['question'], row['irrelevant_context']], label=0)) # 서로 관련 없음.

In [ ]:
# 임베딩 모델 성능 평가.
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

evaluator = EmbeddingSimilarityEvaluator.from_input_examples(examples)
evaluator(sentence_model)

In [ ]:
# MNR(Multiple Negatives Ranking) 손실 함수를 사용해 모델을 미세 조정.

train_samples = []
for idx, row in df_train_ir.iterrows():
    train_samples.append(InputExample(texts=[row['question'], row['context']]))


from sentence_transformers import datasets
batch_size = 16
loader = datasets.NoDuplicatesDataLoader(train_samples, batch_size=batch_size) # 중복 제거.

from sentence_transformers import losses
loss = losses.MultipleNegativesRankingLoss(sentence_model)

epochs = 1
save_path = './klue_mrc_mnr'

# 미세 조정
sentence_model.fit(train_objectives=[(loader, loss)],
                  epochs=epochs,
                  warmup_steps=100,
                  output_path=save_path,
                  show_progress_bar=True)

In [ ]:
# 임베딩 모델 성능 평가.
evaluator(sentence_model)

In [ ]:
# 교차 인코더로 사용할 사전 학습 모델 불러오기.
from sentence_transformers.cross_encoder import CrossEncoder

cross_model = CrossEncoder('klue/roberta-small', num_labels=1) # 많은 계산을 하기 때문에 파라미터 수가 작은 모델 사용.

In [ ]:
# 교차 인코더 성능 평가.
from sentence_transformers.cross_encoder.evaluation import CECorrelationEvaluator

ce_evaluator = CECorrelationEvaluator.from_input_examples(examples)
ce_evaluator(cross_model)

In [ ]:
# 교차 인코더 학습 데이터셋 준비.
train_samples = []
for idx, row in df_train_ir.iterrows():
    train_samples.append(InputExample(texts=[row['question'], row['context']], label=1))
    train_samples.append(InputExample(texts=[row['question'], row['irrelevant_context']], label=0))

In [ ]:
# 교차 인코더 학습.
from torch.utils.data import DataLoader

train_batch_size = 16
num_epochs = 1
model_save_path = 'output/training_mrc'

train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=train_batch_size)
cross_model.fit(train_dataloader=train_dataloader,
                epochs=num_epochs,
                warmup_steps=100,
                output_path=model_save_path)

In [ ]:
ce_evaluator(cross_model)

In [ ]:
from datasets import load_dataset

klue_mrc_test = load_dataset('klue', 'mrc', split='validation')
klue_mrc_test = klue_mrc_test.train_test_split(test_size=1000, seed=42)['test']

In [ ]:
!pip install -qqq faiss-cpu

In [ ]:
import faiss

# 검증 데이터셋 -> 문장 임베딩 변환 -> faiss 인덱스에 저장.
def make_embedding_index(sentence_model, corpus):
    embeddings = sentence_model.encode(corpus)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index

# 검색 쿼리와 유사한 k개 문서 검색.
def find_embedding_top_k(query, sentence_model, index, k):
    embeddings = sentence_model.encode([query])
    distances, indices = index.search(embeddings, k)
    return indices

In [ ]:
import time
import numpy as np

def make_question_context_pairs(question_idx, indices):
    question = klue_mrc_test['question'][int(question_idx)]
    return [[question, klue_mrc_test['context'][int(idx)]] for idx in indices]

# 교차 인코더의 유사도 점수 계산 결과에 따라 순위를 재정렬.
def rerank_top_k(cross_model, question_idx, indices, k):
    indices = np.array(indices, dtype=int)
    input_examples = make_question_context_pairs(question_idx, indices)
    relevance_scores = cross_model.predict(input_examples)
    reranked_indices = indices[np.argsort(relevance_scores)[::-1]] # 유사도 점수가 높은 순으로 인덱스 재정렬.
    return reranked_indices[:k]

# 적중률(성능 평가)
def evaluate_hit_rate(datasets, embedding_model, index, k=10):
    start_time = time.time()
    predictions = []

    # 각 질문별로 k개의 유사 문서 검색.
    for question in datasets['question']:
        topk = find_embedding_top_k(question, embedding_model, index, k)[0]
        predictions.append([int(x) for x in topk])

    # 검색 결과 내에 정답 데이터가 포함돼 있는 경우 정답을 맞췄다고 계산.
    total_prediction_count = len(predictions)
    hit_count = 0
    contexts = datasets['context']

    for idx, prediction in enumerate(predictions):
        for pred in prediction:
            if contexts[int(pred)] == contexts[int(idx)]:
                hit_count += 1
                break

    end_time = time.time()
    return hit_count / total_prediction_count, end_time - start_time

In [ ]:
# 임베딩 모델 평가: 적중률, 걸린 시간 반환.
from sentence_transformers import SentenceTransformer

# 1. 기본 임베딩 모델
base_embedding_model = SentenceTransformer('shangrilar/klue-roberta-base-klue-sts')
base_index = make_embedding_index(base_embedding_model, klue_mrc_test['context']) # 벡터 검색에 사용할 인덱스 생성.
evaluate_hit_rate(klue_mrc_test, base_embedding_model, base_index, 10)

In [ ]:
# 2. 미세 조정한 임베딩 모델
finetuned_embedding_model = SentenceTransformer('shangrilar/klue-roberta-base-klue-sts-mrc')
finetuned_index = make_embedding_index(finetuned_embedding_model, klue_mrc_test['context'])
evaluate_hit_rate(klue_mrc_test, finetuned_embedding_model, finetuned_index, 10)

In [ ]:
# 3. 미세 조정한 임베딩 모델 + 교차 인코더
import time
import numpy as np
from tqdm.auto import tqdm

def evaluate_hit_rate_with_rerank(
    datasets,
    embedding_model,
    cross_model,
    index,
    bi_k=30,      # 추출할 상위 문서수.
    cross_k=10
):
    start_time = time.time()
    predictions = []

    for question_idx, question in enumerate(tqdm(datasets['question'])):
        indices = find_embedding_top_k(question, embedding_model, index, bi_k)[0]
        indices = [int(i) for i in indices]
        reranked = rerank_top_k(cross_model, question_idx, indices, k=cross_k) # 순위 재정렬.
        predictions.append([int(i) for i in reranked])

    total_prediction_count = len(predictions)
    hit_count = 0
    contexts = datasets['context']

    for idx, prediction in enumerate(predictions):
        for pred in prediction:
            if contexts[int(pred)] == contexts[int(idx)]:
                hit_count += 1
                break

    end_time = time.time()
    return hit_count / total_prediction_count, end_time - start_time, predictions

In [ ]:
hit_rate, cosumed_time, predictions = evaluate_hit_rate_with_rerank(klue_mrc_test, finetuned_embedding_model, cross_model, finetuned_index, bi_k=30, cross_k=10)
hit_rate, cosumed_time # 히트율은 제일 높지만, 교차 인코더 때문에 시간 증가.